In [1]:
import torch
import numpy as np
import torchvision
import torchvision.transforms as transforms

## Fixed point module path
***
- Set the relative path to the following module

In [2]:
import sys
sys.path.append('../../../')
import ModelFxp12 as fxp

## CNN Model
***
- Put the model architecture here

In [3]:
class NetCIFAR10(torch.nn.Module):
    def __init__(self):
        torch.nn.Module.__init__(self)

        self.conv1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 26, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2)  # output: 26, 16, 16
        )

        self.conv2 = torch.nn.Sequential(
            torch.nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2)  # output: 26, 8, 8
        )

        self.conv3 = torch.nn.Sequential(
            torch.nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2)  # output: 26, 4, 4
        )

        self.fc = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(26 * 4 * 4, 10)
        )

    def forward(self, xb):
        xb = self.conv1(xb)
        #print(xb.shape)
        xb = self.conv2(xb)
        #print(xb.shape)
        xb = self.conv3(xb)
        #print(xb.shape)
        xb = self.fc1(xb)

        return xb

## Fixed point model
***
- Create here the fixed point version of the CNN model
- Parameters from pretrained models can be read as follows:
```python
cnn_model.load_state_dict(torch.load('parameters.pt', map_location=torch.device('cpu')))
```

In [4]:
weight_shift = 7
bias_shift = 7

cnn_model = NetCIFAR10()

cnn_model.load_state_dict(torch.load('SVHN_qat_sh6.pt', map_location=torch.device('cpu')))

layers = [
    # conv1
    fxp.Conv(cnn_model.conv1[0], weight_shift, bias_shift),
    fxp.ReLU(),
    fxp.Conv(cnn_model.conv1[2], weight_shift, bias_shift),
    fxp.ReLU(),
    fxp.MaxPool2d(2),

    # conv2
    fxp.Conv(cnn_model.conv2[0], weight_shift, bias_shift),
    fxp.ReLU(),
    fxp.Conv(cnn_model.conv2[2], weight_shift, bias_shift),
    fxp.ReLU(),
    fxp.MaxPool2d(2),
    
    # conv3
    fxp.Conv(cnn_model.conv3[0], weight_shift, bias_shift),
    fxp.ReLU(),
    fxp.Conv(cnn_model.conv3[2], weight_shift, bias_shift),
    fxp.ReLU(),
    fxp.MaxPool2d(2),
    
    # fc1
    fxp.Linear(cnn_model.fc[1], weight_shift, bias_shift)       
]

cnn_model_fxp = fxp.ModelFxp(layers)

layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear


## Input generation
***
- Generates an input according to the CNN input dimensions. 

In [5]:
batch = 1
channels = 1
height = 32
width = 32
input = torch.rand(batch, channels , height, width)

out = cnn_model_fxp(input) # 'input' scaled by the model, inside Forward() function

vhex = np.vectorize(hex)
print(f"Model output: {vhex(out.astype('uint32'))}")

Model output: [['0xffffff23' '0xfffffff9' '0x60' '0xc0' '0x1e' '0xffffffd5'
  '0xfffffffa' '0x38' '0xffffffde' '0xffffff86']]


## Input memory image generation for VHDL simulation
***
- Scales the input and converts to integer
- Generates 'input_int.txt' file used as memory image for VHDL simulation

In [6]:
input_int = (input * (2 ** bias_shift)).numpy() # Scales the input and converts to integer
input_int = input_int.astype('uint8')

file = open("input_int.txt", "w+")

hex_chars = 3 # Number of hex characters in each file line. Set according to memory width 
              # memory width = hex_chars * 4

for p in input_int.flatten():    
    file.write('{0:0{1}X}\n'.format(p, hex_chars))

file.close()

## VHDL hardwired parameters generation
***
- Generates
    1. SkyNet_pkg.vhd file
    2. 'raw' directory with the memory images of parameter memories
- Function *GenerateHardwiredParametersVHDL()* requires an input according to the CNN input dimensions

In [7]:
cnn_model_fxp.GenerateHardwiredParametersVHDL(input_int, directory='HardwiredParameters_VHDL')

*** CONVOLUTIONAL LAYERS ***

sortedConvLayers (index, filters): [(0, 26), (1, 26), (2, 26), (3, 26), (4, 26), (5, 26)]
self.maxFilters: 26, convLayer[0]

FILTER_WEIGHTS[0]: begin of convolution layer 0 filters
FILTER_WEIGHTS[2]: convolution layer 0 filter bias
FILTER_WEIGHTS[3]: begin of convolution layer 1 filters
FILTER_WEIGHTS[33]: convolution layer 1 filter bias
FILTER_WEIGHTS[34]: begin of convolution layer 2 filters
FILTER_WEIGHTS[64]: convolution layer 2 filter bias
FILTER_WEIGHTS[65]: begin of convolution layer 3 filters
FILTER_WEIGHTS[95]: convolution layer 3 filter bias
FILTER_WEIGHTS[96]: begin of convolution layer 4 filters
FILTER_WEIGHTS[126]: convolution layer 4 filter bias
FILTER_WEIGHTS[127]: begin of convolution layer 5 filters
FILTER_WEIGHTS[157]: convolution layer 5 filter bias

Data width: 32
Required filter memories depth: 158 (5056 bits). BRAM depth: 256 (8192 bits) (FILTERS_ADDR_WIDTH = 8)
Total required memory: 26 x 5056 = 131456 bits. Total BRAM: 26 x 8192 = 2

## VHDL reconfigurable parameters generation
***
- Generates
    1. ReCoNNet_pkg.vhd file
    2. 'raw' directory with the memory images of parameter memories
    3. CNN_params.txt file with the reconfigurable CNN parameters
- Function *GenerateReCoNNetParametersVHDL()* requires an input according to the CNN input dimensions

In [8]:
cnn_model_fxp.GenerateReCoNNetParametersVHDL(input_int, directory='ReCoNNetParameters_VHDL')

*** CONVOLUTIONAL LAYERS ***

sortedConvLayers (index, filters): [(0, 26), (1, 26), (2, 26), (3, 26), (4, 26), (5, 26)]
self.maxFilters: 26, convLayer[0]

FILTER_WEIGHTS[0]: begin of convolution layer 0 filters
FILTER_WEIGHTS[2]: convolution layer 0 filter bias
FILTER_WEIGHTS[3]: begin of convolution layer 1 filters
FILTER_WEIGHTS[33]: convolution layer 1 filter bias
FILTER_WEIGHTS[34]: begin of convolution layer 2 filters
FILTER_WEIGHTS[64]: convolution layer 2 filter bias
FILTER_WEIGHTS[65]: begin of convolution layer 3 filters
FILTER_WEIGHTS[95]: convolution layer 3 filter bias
FILTER_WEIGHTS[96]: begin of convolution layer 4 filters
FILTER_WEIGHTS[126]: convolution layer 4 filter bias
FILTER_WEIGHTS[127]: begin of convolution layer 5 filters
FILTER_WEIGHTS[157]: convolution layer 5 filter bias

Data width: 32
Required filter memories depth: 158 (5056 bits). BRAM depth: 256 (8192 bits) (FILTERS_ADDR_WIDTH = 8)
Total required memory: 26 x 5056 = 131456 bits. Total BRAM: 26 x 8192 = 2

## C parameters generation
***
- Generates
    1. input.h file containing the input values scaled according to 'bias_shift'
    2. parameters.h file with the CNN parameters
- Function *GenerateParametersC()* requires an **scaled** input according to the CNN input dimensions

In [9]:
cnn_model_fxp.GenerateParametersC(input_int[0], directory='SoftwareParameters_C')

C parameters successfully generated!!!
	- parameters.h
	- input.h

Estimated data memory requirements:
	RAM
		- array parameters[10]: 560 Bytes
		- shared_fmap: 60112 Bytes
		- array acc[26] (LayerConvolution): 52 Bytes
		- array neurons[10] (FullConnected): 20 Bytes
		- input: 2048 Bytes (input_type: int16_t)
		- Total: 62792 bytes (61.3203125 KiB)
	ROM
		-total_conv_weights: 30654 Bytes
		-total_conv_bias: 312 Bytes
		-total_fc_weights: 4160 Bytes
		-total_fc_bias: 20 Bytes
		- Total: 35146 bytes (34.322265625 KiB)
